<div align="center">
  <img src="https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/img/wsu_logo_horizontal.png" alt="Wayne State University Logo" width="320">
  <h1>Lab 6: Model Validation and Generalization</h1>
</div>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WSU-AI-in-ME/ai-in-me-1/blob/main/labs/week06/lab06_model_validation_and_generalization.ipynb)

[Course Repository](https://github.com/WSU-AI-in-ME/ai-in-me-1)

[Lab Index](https://github.com/WSU-AI-in-ME/ai-in-me-1/blob/main/Lab_index.ipynb)

### ME 5995: AI in Mechanical Engineering I: Fundamentals of Manufacturing Data Science
**Wayne State University**

**Individual graded Lab**

**Version: Student Notebook**

Open in Colab and save a personal copy before editing.

## Student Information

Double-click this Markdown cell to edit.

**Student name:** TODO: Enter your name  
**WSU AccessID:** TODO: Enter your AccessID

## Overview

Can a model that predicts other cuts from familiar cutters also predict an unseen cutter?
Use the same data and classifier, but change the validation design.

By the end, you can:
- distinguish row-based CV from cutter-grouped CV;
- inspect actual and predicted labels on training and held-out cuts;
- explain why preprocessing must learn from training rows only.

**Your work:** change two settings and complete four short checkpoints.
Run cells in order. Unchanged example settings run, but do not complete the required experiments.
Work independently after the lecture. Supplied helper and plotting code is for reference;
focus on the settings, outputs, and checkpoint questions, not memorizing every line.

## Data and setup

Colab supplies the required packages. Load the existing PHM feature table directly below.
One row is one machining cut: **315 cuts each from c1, c4, and c6** (945 total).
Use the familiar 11 sensor summaries: force (N), vibration (g), and AE-RMS (V).

| Current-cut category | Mean wear (µm) |
|---|---|
| 0: Slight | ≤75 |
| 1: Intermediate | >75 and <150 |
| 2: Severe | ≥150 |

These are course-defined categories, not official PHM classes or replacement limits.
c6 was examined in prior course work; here it participates in CV, not a new untouched final test.
[Features, processing, and attribution](https://github.com/WSU-AI-in-ME/ai-in-me-1/blob/main/data/phm2010/features/README.md).

In [ ]:
# Provided setup. Run unchanged; comments explain unfamiliar operations.
import numpy as np  # Arrays and numerical comparisons.
import pandas as pd  # Tables and row/column selection.
import matplotlib.pyplot as plt  # Plots of the actual and predicted labels.
from matplotlib.lines import Line2D  # Shared legend symbols for the prediction plot.
from IPython.display import display  # Display readable tables in the notebook.
# StratifiedKFold balances classes; LeaveOneGroupOut separates physical cutters.
from sklearn.model_selection import StratifiedKFold, LeaveOneGroupOut
from sklearn.pipeline import make_pipeline  # Combine scaling with classification.
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, confusion_matrix, ConfusionMatrixDisplay

# Read the CSV into a DataFrame: rows are cuts, columns contain features and metadata.
data = pd.read_csv('https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/data/phm2010/features/phm2010_features.csv')
# isin() keeps these cutter IDs; copy() creates our own working table.
development = data[data['cutter_id'].isin(['c1', 'c4', 'c6'])].copy()
# Sort for reproducible row order and later within-cutter plots.
development = development.sort_values(['cutter_id', 'cut_number'])
# np.where selects values by a condition; these rules check the current labels.
expected_labels = np.where(development['wear_mean_um'] <= 75, 0,
                          np.where(development['wear_mean_um'] < 150, 1, 2))
# to_numpy() extracts values; array_equal() compares every stored label with its rule.
if not np.array_equal(development['wear_level'].to_numpy(), expected_labels):
    raise ValueError('Outdated wear labels. Reload the current course CSV.')
# Each cutter/cut pair must be unique, with 315 cuts in each group.
# value_counts() counts rows per cutter; reindex() requests all three expected IDs.
counts = development['cutter_id'].value_counts().reindex(['c1', 'c4', 'c6'])
if len(development) != 945 or not (counts == 315).all():
    raise ValueError('Expected 315 cuts each from c1, c4, and c6.')
if development.duplicated(['cutter_id', 'cut_number']).any():
    raise ValueError('Duplicate cutter/cut pairs. Reload the course CSV.')

# These columns are inputs X. Wear values, labels, IDs, and cut numbers are excluded.
# mean/sd describe a signal's full-record mean/sample standard deviation.
features = ['force_x_mean', 'force_x_sd', 'force_y_mean', 'force_y_sd',
            'force_z_mean', 'force_z_sd', 'vibration_x_sd', 'vibration_y_sd',
            'vibration_z_sd', 'ae_rms_mean', 'ae_rms_sd']
# The target y is the class label, not continuous wear in micrometers.
labels = development['wear_level']
groups = development['cutter_id']  # Physical grouping metadata, not model inputs.
# Count observations in each cutter/class combination before comparing scores.
display(pd.crosstab(groups, labels))

## 1. Guided example: ordinary three-fold CV

Ordinary CV below shuffles rows while approximately preserving class proportions.
Each fold has **630 training cuts and 315 held-out cuts**, with cutters shared across both sets.
Cutter-grouped CV instead keeps an entire cutter out of fitting, using the same counts.

![Ordinary and cutter-grouped three-fold CV membership](https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/img/lab06/cv_split_comparison.png)

The next function supplies splitting, fitting, and scoring. Run it unchanged;
you do not need to write functions or loops. Each fold uses a fresh Pipeline.
Each fold uses a separately fitted model because its training rows differ.
In cutter-grouped CV, its training cutters also differ.

In [ ]:
# Provided function: split_mode chooses how rows are assigned to folds.
def evaluate_cv(split_mode):
    if split_mode == 'row':
        # Ordinary stratified CV balances class proportions, not physical groups.
        # shuffle mixes the rows; random_state=42 keeps the assignment reproducible.
        splitter = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
        # Save each fold's two arrays of row positions for later inspection.
        split_rows = list(splitter.split(development, labels))
    elif split_mode == 'cutter':
        # Each cutter takes one turn as the held-out group: c1, then c4, then c6.
        # With three cutters, LeaveOneGroupOut produces exactly three folds.
        splitter = LeaveOneGroupOut()
        # groups supplies cutter identity to the splitter, not to the classifier.
        split_rows = list(splitter.split(development, labels, groups))
    else:
        raise ValueError("Choose 'row' or 'cutter'.")

    # Supplied lists retain scores, fitted models, and row positions for later plots.
    training_scores = []
    evaluation_scores = []
    evaluation_cutters = []
    models = []
    # Each loop iteration uses one pair of training/evaluation row-position arrays.
    for train_positions, valid_positions in split_rows:
        # iloc uses row POSITIONS supplied by the splitter, not table index labels.
        fold_train = development.iloc[train_positions]
        fold_valid = development.iloc[valid_positions]
        # StandardScaler learns a mean and standard deviation for each input feature.
        # These scaling statistics are across training cuts, unlike a signal's sd feature.
        # Pipeline.fit learns both scaling and classification from fold_train only.
        # C=1 fixes inverse regularization strength; max_iter is a solver limit.
        # Recreate the Pipeline each time so no learning carries between folds.
        model = make_pipeline(StandardScaler(),
                              LogisticRegression(C=1, solver='lbfgs', max_iter=2000))
        # First argument: the 11 input columns. Second argument: their known labels.
        model.fit(fold_train[features], fold_train['wear_level'])
        # Predicting training rows describes fit; it is not a generalization estimate.
        pred_train = model.predict(fold_train[features])
        # Evaluation reuses the fitted scaler and model; it never calls fit again.
        pred_valid = model.predict(fold_valid[features])
        # Actual labels come first. Macro-F1 weights the three classes equally.
        # labels=[0,1,2] retains all classes, including any the model never predicts.
        # zero_division=0 reports an undefined class F1 as zero by convention.
        train_f1 = f1_score(fold_train['wear_level'], pred_train,
                           labels=[0, 1, 2], average='macro', zero_division=0)
        valid_f1 = f1_score(fold_valid['wear_level'], pred_valid,
                           labels=[0, 1, 2], average='macro', zero_division=0)
        # append() adds one value to each list; their positions match the fold order.
        training_scores.append(train_f1)
        evaluation_scores.append(valid_f1)
        # unique() lists cutter names; join() formats them for the output table.
        evaluation_cutters.append(', '.join(sorted(fold_valid['cutter_id'].unique())))
        models.append(model)  # Keep this fitted model for inspection, without refitting.
    # Supplied reporting code: one row per fold, with both kinds of score.
    results = pd.DataFrame({'Fold': [1, 2, 3], 'Evaluation cutters': evaluation_cutters,
                            'Training macro-F1': training_scores,
                            'Evaluation macro-F1': evaluation_scores})
    # Return the table plus models and row positions needed to inspect a chosen fold.
    return results, models, split_rows

# Completed example. Keep the ordinary-CV results for the next comparison.
row_results, row_models, row_splits = evaluate_cv('row')
# round(3) changes display precision only; fitting and scoring retain full precision.
display(row_results.round(3))

### CP1: Identify the evaluation target

Does ordinary CV evaluate **other cuts of observed cutters** or **a cutter absent from training**? Choose one.

**Your response:** TODO.

## 2. Change the splitting rule

Change `student_split` from **'row' to 'cutter'** and run the cell. Keep the
feature list and model settings fixed. Read each fold before comparing the means.

In [ ]:
# Required change 1: hold out one whole cutter instead of a mixture of rows.
student_split = 'row'  # TODO: Change to 'cutter'.
# This runs the same fitting/scoring procedure with a different split rule.
student_results, student_models, student_splits = evaluate_cv(student_split)
# Check the Evaluation cutters column to see what was held out in each fold.
display(student_results.round(3))
# Average fold scores, not pooled predictions. Higher macro-F1 is better.
row_mean = row_results['Evaluation macro-F1'].mean()
student_mean = student_results['Evaluation macro-F1'].mean()
display(pd.DataFrame({'Split': ['Ordinary three-fold', student_split + ' three-fold'],
                      'Mean evaluation macro-F1': [row_mean, student_mean]}).round(3))

### CP2: Interpret the comparison

In **up to two sentences**, choose which CV result you would report for use on an unseen cutter. Cite both mean evaluation scores and explain your choice.

**Your response:** TODO.

## 3. Inspect actual and predicted labels

After setting `student_split = 'cutter'`, first keep `inspection_fold = 1` and run
the next three code cells. Then change it to **3** and rerun those same three cells:
selection, prediction plot, and confusion matrices. Fold 2 is optional to inspect.
The comparison table and left matrix always retain fold 1 as a reference, even after Run all.
The three-panel prediction plot uses only your selected fold's fitted model.
Finish with fold 3 selected. You do not need screenshots of earlier runs.

Black lines mark actual labels; green dots are training predictions; orange crosses
are held-out predictions. Red rings mark mistakes. Cut number orders observations
for viewing only; it is not a predictor, and these are not future-wear forecasts.

In [ ]:
# First run with 1. Then change to 3 and rerun these three code cells:
# this selection cell, the prediction plot, and the confusion matrices.
inspection_fold = 1  # TODO: After the first run, change 1 to 3.
if inspection_fold not in [1, 2, 3]:
    raise ValueError('Choose fold 1, 2, or 3.')
# Python lists start at position 0, so subtract 1 from the displayed fold number.
selected_model = student_models[inspection_fold - 1]
train_positions, valid_positions = student_splits[inspection_fold - 1]
selected_train = development.iloc[train_positions].copy()
selected_valid = development.iloc[valid_positions].copy()
# Copies let us add plotting columns without changing the original feature table.
# predict() reuses the selected model. Neither line changes its learned parameters.
selected_train['predicted_label'] = selected_model.predict(selected_train[features])
selected_valid['predicted_label'] = selected_model.predict(selected_valid[features])
# Compare each prediction with the matching row's actual class.
# != produces True for a mistake and False for a correct prediction.
selected_train['incorrect'] = selected_train['predicted_label'] != selected_train['wear_level']
selected_valid['incorrect'] = selected_valid['predicted_label'] != selected_valid['wear_level']
print('Selected split:', student_split, '| Fold:', inspection_fold)
print('Training cutters:', ', '.join(sorted(selected_train['cutter_id'].unique())))
print('Evaluation cutters:', ', '.join(sorted(selected_valid['cutter_id'].unique())))
# Keep a fixed reference so changing the selected fold never erases the comparison.
# iloc[[...]] selects table rows; drop_duplicates avoids showing fold 1 twice.
fold_comparison = student_results.iloc[[0, inspection_fold - 1]].drop_duplicates()
display(fold_comparison.round(3))

In [ ]:
# Provided plotting code. No new fitting, settings, or written answers are required.
# Show one panel per cutter, with identical axes so class transitions can be compared.
# sharex/sharey use the same cut-number and category scales across all three panels.
fig, axes = plt.subplots(3, 1, figsize=(10, 6.6), sharex=True, sharey=True)
cutter_names = ['c1', 'c4', 'c6']
for panel in range(3):
    cutter = cutter_names[panel]
    # Keep the actual cut order within this cutter, not across different cutters.
    actual_rows = development[development['cutter_id'] == cutter].sort_values('cut_number')
    training_rows = selected_train[selected_train['cutter_id'] == cutter]
    evaluation_rows = selected_valid[selected_valid['cutter_id'] == cutter]
    # A Boolean mask selects the incorrect predictions to highlight with red rings.
    training_errors = training_rows[training_rows['incorrect']]
    evaluation_errors = evaluation_rows[evaluation_rows['incorrect']]
    ax = axes[panel]  # Choose this cutter's panel.
    # step() shows the discrete actual category; it is not a continuous wear estimate.
    ax.step(actual_rows['cut_number'], actual_rows['wear_level'], where='mid',
            color='black', linewidth=1.6, zorder=1)
    # scatter() places predictions at their cut numbers; s controls marker size.
    # Green dots describe fit on training data. Orange x marks describe held-out data.
    ax.scatter(training_rows['cut_number'], training_rows['predicted_label'],
               s=12, color='seagreen', alpha=0.65, zorder=2)
    ax.scatter(evaluation_rows['cut_number'], evaluation_rows['predicted_label'],
               s=18, marker='x', color='darkorange', zorder=3)
    # Hollow red circles emphasize errors without hiding the prediction underneath.
    # zorder draws these circles above the other layers.
    ax.scatter(training_errors['cut_number'], training_errors['predicted_label'],
               s=32, facecolors='none', edgecolors='crimson', linewidths=0.8, zorder=4)
    ax.scatter(evaluation_errors['cut_number'], evaluation_errors['predicted_label'],
               s=32, facecolors='none', edgecolors='crimson', linewidths=0.8, zorder=4)
    # Counts in each panel distinguish a shared cutter from a fully held-out cutter.
    ax.set_title(cutter + ': training ' + str(len(training_rows)) +
                 ', held-out ' + str(len(evaluation_rows)), loc='left', fontsize=11)
    ax.set_yticks([0, 1, 2], labels=['0 Slight', '1 Intermediate', '2 Severe'])
    ax.set_ylim(-0.25, 2.25)
    ax.set_xlim(1, 315)
    ax.grid(alpha=0.18)
# Shared legend: empty handles explain the plotting symbols once for all panels.
legend_items = [Line2D([], [], color='black', label='Actual label'),
                Line2D([], [], marker='o', linestyle='', color='seagreen', label='Training prediction'),
                Line2D([], [], marker='x', linestyle='', color='darkorange', label='Held-out prediction'),
                Line2D([], [], marker='o', linestyle='', markerfacecolor='none',
                       color='crimson', label='Incorrect prediction')]
fig.legend(handles=legend_items, loc='upper center', ncol=4, fontsize=9)
axes[-1].set_xlabel('Cut number (within each cutter)')
plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()

In [ ]:
# Provided summary for the same held-out rows used in the orange predictions above.
# Actual labels are rows; predicted labels are columns. Off-diagonal cells are errors.
selected_cm = confusion_matrix(selected_valid['wear_level'],
                               selected_valid['predicted_label'], labels=[0, 1, 2])
# Each cell is a cut count, not a probability or a percentage.
# Reference fold 1 is reconstructed from its saved model, with no new fitting.
reference_model = student_models[0]
reference_positions = student_splits[0][1]  # Second array holds evaluation rows.
reference_valid = development.iloc[reference_positions]
reference_predictions = reference_model.predict(reference_valid[features])
reference_cm = confusion_matrix(reference_valid['wear_level'], reference_predictions,
                                labels=[0, 1, 2])
# Identical count scales make the two evaluation matrices comparable.
fig, axes = plt.subplots(1, 2, figsize=(8.4, 3.5))
ConfusionMatrixDisplay(reference_cm, display_labels=[0, 1, 2]).plot(
    ax=axes[0], colorbar=False, cmap='Blues', values_format='d',
    im_kw={'vmin': 0, 'vmax': 315})
ConfusionMatrixDisplay(selected_cm, display_labels=[0, 1, 2]).plot(
    ax=axes[1], colorbar=False, cmap='Blues', values_format='d',
    im_kw={'vmin': 0, 'vmax': 315})
# values_format='d' displays integer counts. Correct predictions lie on the diagonal.
# Show actual held-out cutter names for either split rule.
reference_cutters = ', '.join(sorted(reference_valid['cutter_id'].unique()))
selected_cutters = ', '.join(sorted(selected_valid['cutter_id'].unique()))
axes[0].set_title('Reference: fold 1 (' + reference_cutters + ')')
axes[1].set_title('Selected: fold ' + str(inspection_fold) + ' (' + selected_cutters + ')')
fig.text(0.5, 0.01, 'Classes: 0 Slight, 1 Intermediate, 2 Severe', ha='center')
plt.tight_layout(rect=[0, 0.06, 1, 1])
plt.show()

### CP3: Compare held-out cutters

Answer each item in **one sentence**.

**Held-out cutters and scores:** Name the held-out cutters in folds 1 and 3 and compare their evaluation macro-F1.

**Your response:** TODO.

**Fold 3 error:** Identify its most frequent error using the actual and predicted labels.

**Your response:** TODO.

**Training scores:** Would these scores alone reveal the difference in held-out performance? Explain.

**Your response:** TODO.

**Interpretation limit:** matching the row counts does not make the score gap a
measured leakage effect. Cutter composition and label distributions differ.
Three cutters give limited generalization evidence. Do not tune the model using these results.

## 4. Repair preprocessing leakage

![Training-only scaling and classification within each fold](https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/img/lab06/fold_preprocessing.png)

A colleague fits a scaler on all three cutters before fitting a classifier on c1/c4
and evaluating c6.

### CP4: Correct the workflow

In **one sentence**, identify the rows that should fit the scaler and explain how to process c6.

**Your response:** TODO.

In [ ]:
# This reminder checks settings only; you must also complete the written responses.
required_settings_done = (student_split == 'cutter' and inspection_fold == 3)
if required_settings_done:
    print('Two settings updated. Check CP1 through CP4 and keep the plots visible.')
else:
    print("Still required: student_split = 'cutter' and inspection_fold = 3.")

## Submission checklist

Rendered Markdown checkboxes are not clickable. Edit this cell and change `[ ]` to `[x]`.

- [ ] Enter your name and WSU AccessID; make two setting changes and complete CP1 through CP4.
- [ ] Restart the runtime and Run all. Retain the score tables, prediction plot, and confusion matrix.
- [ ] Submit `Lab06_Firstname_Lastname.ipynb` and matching `.pdf` individually through Canvas.

No separate report. Provided code and optional work are not separately graded.

## References

- PHM Society, [2010 PHM Society Conference Data Challenge](https://phmsociety.org/phm_competition/2010-phm-society-conference-data-challenge/).
- Course [feature dictionary](https://github.com/WSU-AI-in-ME/ai-in-me-1/blob/main/data/phm2010/features/DATA_DICTIONARY.md) and [source/rights record](https://github.com/WSU-AI-in-ME/ai-in-me-1/blob/main/data/phm2010/README.md).
- scikit-learn: [cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html) and [data leakage](https://scikit-learn.org/stable/common_pitfalls.html).